In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn

# Helper functions

In [5]:
def loadData(dataset, traceLength):
    
    path1 = "Data/CSV/"
    path2 = ["MalwarePlusClean", "MalwarePlusClean2", "AllMalware", "AllMalware2", "AllMalwarePlusClean", "AllMalwarePlusClean2"]
    path3_2 = ["010_2.csv", "020_2.csv", "040_2.csv", "060_2.csv", "080_2.csv", "100_2.csv", "200_2.csv", "400_2.csv", "600_2.csv", "800_2.csv", "1000_2.csv"]
    path3_5 = ["010_5.csv", "020_5.csv", "040_5.csv", "060_5.csv", "080_5.csv", "100_5.csv", "200_5.csv", "400_5.csv", "600_5.csv", "800_5.csv", "1000_5.csv"]
    path3_6 = ["010_6.csv", "020_6.csv", "040_6.csv", "060_6.csv", "080_6.csv", "100_6.csv", "200_6.csv", "400_6.csv", "600_6.csv", "800_6.csv", "1000_6.csv"]
    path3 = [path3_2, path3_5, path3_6]

    filename = f"{path1}{path2[dataset]}/{path3[dataset//2][traceLength]}"

    data = pd.read_csv(filename, header=None).values

    X = data.transpose()[:-1].transpose()
    y = data.transpose()[-1].transpose()

    return filename, X, y

In [13]:
def runCV(clf, X, y):
    scoring = ["accuracy"]

    scores = sklearn.model_selection.cross_validate(clf, X, y, cv=10, scoring=scoring, n_jobs=-1)

    return scores

In [ ]:
def saveResults(method, dataset, variables, scores):
    with open("output.txt", "a") as file:
        file.write(f"{method}:\n\tDataset: {dataset}\n\tVariables: {variables}\n\n\tAccuracy: {scores["test_accuracy"].mean()}\n\n\n")

# Choose a dataset

In [ ]:
# Possible datasets:
# MalwarePlusClean = 0
# MalwarePlusClean2 = 1
# AllMalware = 2
# AllMalware2 = 3
# AllMalwarePlusClean = 4
# AllMalwarePlusClean2 = 5

chosenDataset = 0

# Naive Bayes

In [ ]:
clf = sklearn.naive_bayes.ComplementNB()

for i in range(11):

    dataset, X, y = loadData(chosenDataset, i)

    scores = runCV(clf, X, y)

    saveResults("Naive Bayes", dataset, "", scores)

# Logistic regression

In [ ]:
l1_values = [0, 0.25, 0.5, 0.75, 1]
C_values = [0.001, 0.01, 0.1, 1, 10]

for l1 in l1_values:
    for C in C_values:

        clf = sklearn.linear_model.LogisticRegression(solver="saga", penalty="elasticnet", l1_ratio=l1, C=C)

        for i in range(11):

            dataset, X, y = loadData(chosenDataset, i)

            scores = runCV(clf, X, y)

            saveResults(method="Logistic regression", dataset=dataset, variables=f"C = {C}, Penalty = elasticnet, l1_ratio = {l1}", scores=scores)

# K-nearest neighbor

In [ ]:
neighbors = [5, 7, 9, 11, 13, 15]
weights = ["uniform", "distance"]
metrics = ["minkowski", "euclidean", "manhattan"]

for w in weights:
    for m in metrics:
        for n in neighbors:

            clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors=n, weights=w, metric=m)

            for i in range(11):

                dataset, X, y = loadData(chosenDataset, i)

                scores = runCV(clf, X, y)

                saveResults(method="K-nearest neighbor", dataset=dataset, variables=f"Number of neighbors = {n}, Weight = {w}, Metric = {m}", scores=scores)

# Decision tree

In [ ]:
criterion = ["gini", "entropy"]
min_sample_split = [2, 10, 100]
min_sample_leaf = [1, 10, 100]
max_depth = [5, 10, 20, None]

for c in criterion:
    for s in min_sample_split:
        for l in min_sample_leaf:
            for m in max_depth:

                clf = sklearn.tree.DecisionTreeClassifier(criterion=c, max_depth=m, min_samples_leaf=l, min_samples_split=s)

                for i in range(11):

                    dataset, X, y = loadData(chosenDataset, i)

                    scores = runCV(clf, X, y)

                    saveResults(method="Decision tree", dataset=dataset, variables=f"Criterion = {c}, Max depth = {m}, Min sample split = {s}, Min sample leaf = {l}", scores=scores)

# Random forest

In [ ]:
n_estimators = [100, 200, 500]
criterion = ["gini", "entropy"]
min_sample_split = [2, 10]
min_sample_leaf = [1, 10]
max_depth = [10, 20, 50, None]

for c in criterion:
    for s in min_sample_split:
        for l in min_sample_leaf:
            for m in max_depth:
                for n in n_estimators:

                    clf = sklearn.ensemble.RandomForestClassifier(n_estimators=n, criterion=c, max_depth=m, min_samples_leaf=l, min_samples_split=s)

                    for i in range(11):

                        dataset, X, y = loadData(chosenDataset, i)

                        scores = runCV(clf, X, y)

                        saveResults(method="Random forest", dataset=dataset, variables=f"Number of trees = {n}, Criterion = {c}, Max depth = {m}, Min sample split = {s}, Min sample leaf = {l}", scores=scores)

# Get full results from chosen configurations

Input the relevant configuration for the relevant model to generate the more detailed results

In [ ]:
# Results part 1

scoring = ["accuracy", "f1_weighted", "roc_auc_ovo_weighted"]

#clf = sklearn.tree.DecisionTreeClassifier(criterion="entropy", max_depth=20, min_samples_leaf=1, min_samples_split=2)
#clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors=5, weights="distance", metric="manhattan")
#clf = sklearn.linear_model.LogisticRegression(solver="saga", penalty="elasticnet", l1_ratio=0.25, C=0.001)
#clf = sklearn.naive_bayes.ComplementNB()
#clf = sklearn.ensemble.RandomForestClassifier(n_estimators=100, criterion="entropy", max_depth=20, min_samples_leaf=1, min_samples_split=2)

dataset, X, y = loadData(chosenDataset, 0)

scores = sklearn.model_selection.cross_validate(clf, X, y, cv=10, scoring=scoring, n_jobs=-1)

print(f"Mean accuracy: {scores["test_accuracy"].mean()*100}\nStandard deviation accuracy: {np.std(scores["test_accuracy"])*100}\nWeighted F1-score: {scores["test_f1_weighted"].mean()}\nAUC: {scores["test_roc_auc_ovo_weighted"].mean()}\nMean training time: {scores["fit_time"].mean()}\nStandard deviation training time: {np.std(scores["fit_time"])}\nMean testing time: {scores["score_time"].mean()}\nStandard deviation testing time: {np.std(scores["score_time"])}")

In [ ]:
# Results part 2

# The confusion matrix has to be filled out manually in order to calculate the recall, false positive rate and false negative rate.

#confusion_matrix = np.matrix([[0, 0], [0, 0]])
confusion_matrix = np.matrix([[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]])

FP = confusion_matrix.sum(axis=0) - np.diag(confusion_matrix)  
FN = confusion_matrix.sum(axis=1).transpose() - np.diag(confusion_matrix)
TP = np.diag(confusion_matrix)
TN = confusion_matrix.sum() - (FP + FN + TP)

TPR = TP/(TP+FN)
FPR = FP/(FP+TN)
FNR = FN/(TP+FN)

print("Recall:", TPR)
print("False positive rate:", FPR)
print("False negative rate:", FNR)

In [ ]:
# ROC curve

dataset, X, y = loadData(0, 0)

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, random_state=0, test_size=0.1)

#clf = sklearn.tree.DecisionTreeClassifier(criterion="entropy", max_depth=20, min_samples_leaf=1, min_samples_split=2).fit(X_train, y_train)
#clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors=5, weights="distance", metric="manhattan").fit(X_train, y_train)
#clf = sklearn.linear_model.LogisticRegression(solver="saga", penalty="saga", l1_ratio=0.75, C=0.1).fit(X_train, y_train)
#clf = sklearn.naive_bayes.ComplementNB().fit(X_train, y_train)
#clf = sklearn.ensemble.RandomForestClassifier(n_estimators=500, criterion="gini", max_depth=50, min_samples_leaf=1, min_samples_split=2).fit(X_train, y_train)

sklearn.metrics.RocCurveDisplay.from_estimator(clf, X_test, y_test, name=None, pos_label=None)

plt.show()

In [ ]:
# Confusion matrix

dataset, X, y = loadData(chosenDataset, 0)

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, random_state=0, test_size=0.1)

#clf = sklearn.tree.DecisionTreeClassifier(criterion="entropy", max_depth=20, min_samples_leaf=1, min_samples_split=2).fit(X_train, y_train)
#clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors=5, weights="distance", metric="manhattan").fit(X_train, y_train)
#clf = sklearn.linear_model.LogisticRegression(solver="saga", penalty="saga", l1_ratio=0.75, C=0.1).fit(X_train, y_train)
#clf = sklearn.naive_bayes.ComplementNB().fit(X_train, y_train)
#clf = sklearn.ensemble.RandomForestClassifier(n_estimators=500, criterion="gini", max_depth=50, min_samples_leaf=1, min_samples_split=2).fit(X_train, y_train)

sklearn.metrics.ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, display_labels=clf.classes_, cmap="Blues", xticks_rotation="vertical")

plt.show()